# NumPy Vectorization Cheatsheet for Economic Analysis
A quick-reference companion to the Economics Vectorization lab. Organized by task, with the NumPy syntax on the left and a typical economics use case on the right. Run the setup cell once, then jump to any section.

In [ ]:
import numpy as np
import time

# reference data used throughout this cheatsheet
goods = ['Wheat', 'Oil', 'Steel', 'Electronics', 'Services']
p = np.array([200., 350., 500., 800., 150.])   # prices, $
q = np.array([120.,  80.,  60.,  40., 300.])   # quantities, thousand units

countries = ['USA', 'Germany', 'Japan', 'Brazil']
years = [2020, 2021, 2022, 2023, 2024, 2025]
GDP = np.array([[21000., 23000., 25400., 26900., 27900., 29300.],
                [ 3900.,  4300.,  4100.,  4300.,  4500.,  4700.],
                [ 5000.,  4900.,  4200.,  4400.,  4100.,  4200.],
                [ 1400.,  1600.,  1900.,  2100.,  2200.,  2300.]])

## 1. Array / Vector Creation
| Syntax | Result | Economics use case |
|---|---|---|
| `np.zeros(n)` | 1-D array of n zeros | Placeholder price/quantity vector before loading data |
| `np.zeros((m,n))` | m x n array of zeros | Placeholder panel (m countries x n years) |
| `np.ones(n)` | 1-D array of n ones | Uniform weights, e.g. equal portfolio weights |
| `np.arange(n)` | `[0, 1, ..., n-1]` | Index of periods or goods |
| `np.array([...])` | Array from a Python list | Hard-coded prices, quantities, coefficients |
| `np.random.rand(n)` | n uniform random numbers in [0,1) | Monte Carlo simulation of returns/prices |
| `np.random.random_sample(n)` | Same as `rand`, shape-tuple friendly | Simulated shocks |
| `np.random.normal(mu, sigma, n)` | n draws from Normal(mu, sigma) | Simulated income/return shocks |
| `np.linspace(a, b, n)` | n evenly spaced points from a to b | Grid of interest rates or tax rates to sweep over |

In [ ]:
print(np.zeros(5))
print(np.arange(5))
print(np.array([200., 350., 500., 800., 150.]))
print(np.random.normal(0.02, 0.01, 5))   # e.g. simulated monthly returns, mean 2%, sd 1%
print(np.linspace(0.0, 0.10, 5))          # e.g. a grid of interest rates from 0% to 10%

## 2. Indexing
| Syntax | Meaning | Economics use case |
|---|---|---|
| `p[i]` | element at position i | Price of the i-th good |
| `p[-1]` | last element | Most recent period's value in a time series |
| `GDP[i, j]` | element at row i, col j | Country i's GDP in year j |
| `GDP[i]` or `GDP[i, :]` | entire row i (1-D) | All years of GDP for country i |
| `GDP[:, j]` | entire column j (1-D) | All countries' GDP in year j |

In [ ]:
print(p[2])          # price of the 3rd good
print(p[-1])          # price of the last good
print(GDP[1, 3])       # Germany, 2023
print(GDP[0])          # USA, all years
print(GDP[:, 0])       # all countries, 2020

## 3. Slicing
`start:stop:step` on any axis. Omit any part to take a default (start=0, stop=end, step=1).

| Syntax | Meaning | Economics use case |
|---|---|---|
| `p[1:4]` | elements 1,2,3 | A subset of goods in a basket |
| `p[:3]` | first 3 elements | Earliest 3 periods of a time series |
| `p[2:]` | from index 2 to the end | All periods after a policy change |
| `p[::2]` | every other element | Even-indexed years, e.g. biennial data |
| `GDP[:, 2:5]` | all rows, columns 2-4 | All countries, a 3-year window |
| `GDP[1:3, :]` | rows 1-2, all columns | Two countries, all years |

In [ ]:
print(p[1:4])
print(p[::2])
print(GDP[:, 2:5])     # all countries, 2022-2024
print(GDP[1:3, :])     # Germany and Japan, all years

## 4. Single-Array (Reduction) Operations
| Syntax | Result | Economics use case |
|---|---|---|
| `np.sum(a)` | sum of all elements | Total quantity sold; total GDP across countries |
| `np.mean(a)` | average | Average price; average GDP growth |
| `np.std(a)` | standard deviation | Volatility of returns or prices |
| `np.min(a)` / `np.max(a)` | min / max | Cheapest good; peak GDP year |
| `np.argmin(a)` / `np.argmax(a)` | index of min / max | Which good is cheapest; which year had peak GDP |
| `np.cumsum(a)` | running total | Cumulative expenditure over time |
| `np.diff(a)` | first differences | Period-over-period change (e.g. GDP growth in levels) |
| `a**2`, `np.sqrt(a)`, `np.log(a)`, `np.exp(a)` | element-wise math | Quadratic costs; log-GDP for growth-rate regressions |
| `-a` | negation | Flip a surplus vector into a shortage/deficit vector |

In [ ]:
print(np.sum(q))                 # total quantity
print(np.mean(p))                # average price
print(np.std(GDP[0]))            # volatility of USA GDP series
print(np.argmax(GDP[0]))          # index of USA's peak GDP year
print(np.diff(GDP[0]))            # year-over-year GDP change, USA
print(np.log(GDP[0]))             # log GDP, e.g. for a growth-rate regression

## 5. Vector-Vector Element-wise Operations
Both vectors must have the same shape.

| Syntax | Result | Economics use case |
|---|---|---|
| `a + b`, `a - b` | element-wise add/subtract | Price change between two periods |
| `a * b` | element-wise multiply | Revenue per good = price * quantity |
| `a / b` | element-wise divide | Price relative to a base-period price (a simple price index component) |
| `a > b`, `a == b` | element-wise boolean | Which goods got more expensive |
| `np.where(cond, a, b)` | conditional select | Cap prices at a ceiling, floor at a subsidy level |

In [ ]:
p_next_year = np.array([210., 340., 520., 760., 155.])
print(p_next_year - p)                  # price change
print(p * q)                             # revenue by good
print(p_next_year / p)                   # relative price (price index component)
print(p_next_year > p)                   # which goods got more expensive
print(np.where(p > 400, 400, p))         # cap prices at 400

## 6. Scalar-Vector Operations (Broadcasting)
A single number is applied to every element — the basis of inflation adjustment, currency conversion, and tax/subsidy rates.

| Syntax | Result | Economics use case |
|---|---|---|
| `a * k` | scale every element by k | Apply an inflation rate, exchange rate, or tax rate |
| `a + k` | shift every element by k | Add a flat per-unit subsidy to all prices |
| `a / k` | rescale | Convert nominal values to per-capita by dividing by population |

In [ ]:
inflation_rate = 0.03
print(p * (1 + inflation_rate))     # inflation-adjusted prices
print(p * 0.92)                      # convert USD prices to EUR
population_millions = 331
print(GDP[0] / population_millions)  # USA GDP per capita by year ($ billion / million people)

## 7. Dot Product
`np.dot(a, b)` (equivalently `a @ b`) multiplies element-wise and sums — the single most important vector operation in economics: it is *total expenditure*, *total revenue*, or one component of *GDP* in one call.

| Syntax | Result | Economics use case |
|---|---|---|
| `np.dot(p, q)` or `p @ q` | scalar | Total expenditure = sum(price_i * quantity_i) |
| `np.dot(w, x) + b` | scalar | Predicted value from a linear model (demand, wages, prices) |
| `np.dot(A, d)` | vector | Matrix-vector product, e.g. input-output intermediate demand |

In [ ]:
print(np.dot(p, q))     # total expenditure
print(p @ q)             # same thing, @ is the matrix-multiplication operator

w = np.array([-0.8, 0.05, 0.02]); x = np.array([10., 500., 20.]); b = 12.0
print(np.dot(w, x) + b)  # predicted demand from a linear model

## 8. Speed: Vectorize, Don't Loop
Economic datasets (household surveys, tick-level financial data, Monte Carlo simulations) are often huge. Always prefer the vectorized form.

In [ ]:
n = 2_000_000
a = np.random.rand(n); b = np.random.rand(n)

tic = time.time(); c = np.dot(a, b); toc = time.time()
print(f"Vectorized: {1000*(toc-tic):.3f} ms")

tic = time.time()
c = 0.0
for i in range(n):
    c += a[i]*b[i]
toc = time.time()
print(f"Loop:       {1000*(toc-tic):.3f} ms")
del a, b

## 9. Matrix Creation & Shape
| Syntax | Result | Economics use case |
|---|---|---|
| `np.zeros((m,n))` | m x n zeros | Placeholder panel data |
| `np.array([[...],[...]])` | m x n from nested lists | Hard-coded panel/IO table |
| `a.reshape(m, n)` | reshape a 1-D array | Reshape a flat data pull into (countries, years) |
| `a.shape` | tuple of dimensions | Sanity-check before combining datasets |
| `np.eye(n)` | n x n identity matrix | The `I` in Leontief's `(I - A)^-1` |

In [ ]:
print(GDP.shape)
print(np.arange(24).reshape(4, 6).shape)   # e.g. reshape 24 flat data points into 4 countries x 6 years
print(np.eye(3))

## 10. Matrix Indexing & Slicing
| Syntax | Result | Economics use case |
|---|---|---|
| `M[i, j]` | scalar | One country, one year |
| `M[i, :]` or `M[i]` | row as 1-D vector | One country, all years |
| `M[:, j]` | column as 1-D vector | One year, all countries (cross-section) |
| `M[i1:i2, j1:j2]` | sub-matrix | A subset of countries and years |
| `M.T` | transpose | Swap countries/years axes, e.g. before plotting |

In [ ]:
print(GDP[1, 3])         # Germany, 2023
print(GDP[:, -1])         # all countries, most recent year (cross-section)
print(GDP[0:2, 1:4])      # USA & Germany, 2021-2023
print(GDP.T.shape)        # (years, countries) instead of (countries, years)

## 11. Matrix-Vector & Matrix-Matrix Products
| Syntax | Result | Economics use case |
|---|---|---|
| `np.dot(A, d)` or `A @ d` | vector | Input-output intermediate demand; portfolio value = weights @ prices |
| `np.dot(A, B)` or `A @ B` | matrix | Combine two linear transformations, e.g. chained sector multipliers |
| `np.linalg.solve(I - A, d)` | vector | Full Leontief equilibrium output given final demand |
| `np.linalg.inv(M)` | matrix | Explicit inverse (use `solve` when possible — it's faster and more stable) |

In [ ]:
A = np.array([[0.20, 0.10, 0.05],
              [0.15, 0.25, 0.10],
              [0.10, 0.05, 0.20]])
d = np.array([100., 150., 200.])

print(A @ d)                                  # first-round intermediate demand
print(np.linalg.solve(np.eye(3) - A, d))       # full Leontief equilibrium output

## 12. Common Pitfalls
- **Shape mismatch**: element-wise ops (`+`, `-`, `*`, `/`) require identical shapes, unless broadcasting rules apply (one side is a scalar, or a matching size-1 axis).
- **Row vs. column vectors**: `np.array([1,2,3])` has shape `(3,)`, not `(3,1)` or `(1,3)`. Use `.reshape(-1,1)` when you need an explicit column vector, e.g. for matrix multiplication with a 2-D matrix.
- **Integer division surprises**: `np.array([1,2,3]) / 2` gives floats; watch dtypes when mixing integer counts (e.g. population) with money amounts.
- **`np.dot` on 2-D arrays is matrix multiplication**, not element-wise — use `*` for element-wise, `@`/`np.dot` for matrix products.
- **Mutating shared arrays**: slicing returns a *view*, not a copy — modifying a slice can silently change the original panel. Use `.copy()` when you need an independent version.

## Congratulations!
Keep this notebook open as a side reference while building your own vectorized economic models — pair it with the Skeleton lab notebook for guided practice, or the Template notebook to start a new project.